# DAS Policy Crawler (Playwright + PDF extraction)

This notebook:
1. Verifies the active Python interpreter (should be your `.venv`).
2. Installs required dependencies.
3. Collects DAS policy links using Playwright-rendered DOM + network JSON mining.
4. Crawls each policy page, extracts references/PDF text, and writes outputs to:
   - `tools/NoteBooks/out/das_policies/txt`
   - `tools/NoteBooks/out/das_policies/pdf`

In [9]:
import sys
from pathlib import Path

exe = Path(sys.executable).resolve()
print('Python executable:', exe)
print('Python version   :', sys.version.split()[0])

# Robust check: validate interpreter path itself (not notebook cwd)
exe_norm = str(exe).lower().replace('/', '\\')
if '\\.venv\\scripts\\python.exe' in exe_norm:
    print('\n✅ .venv kernel detected.')
else:
    print('\nWARNING: Notebook kernel does not appear to be .venv.')
    print('Pick kernel from policy-back/.venv to ensure dependency isolation.')

Python executable: C:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\.venv\Scripts\python.exe
Python version   : 3.12.10

✅ .venv kernel detected.


In [10]:
import subprocess
import sys

# Install required packages into the ACTIVE notebook interpreter
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-U',
    'requests',
    'beautifulsoup4',
    'pypdf',
    'playwright',
    'lxml',
])

# Install browser runtime required by Playwright
subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])

print('Dependencies installed.')

Dependencies installed.


In [20]:
import os
import re
import json
import time
import hashlib
import asyncio
import sys
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from playwright.sync_api import sync_playwright

START_URL = 'https://das.ohio.gov/home/policy-finder/filter-policy-finder'
BROWSER_UA = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
    '(KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36'
)

# Force output into tools/NoteBooks/out/das_policies
if (Path.cwd() / 'tools' / 'NoteBooks').exists():
    NOTEBOOKS_DIR = Path.cwd() / 'tools' / 'NoteBooks'
else:
    NOTEBOOKS_DIR = Path.cwd()

OUT_DIR = NOTEBOOKS_DIR / 'out' / 'das_policies'
TXT_DIR = OUT_DIR / 'txt'
PDF_DIR = OUT_DIR / 'pdf'

TXT_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

print('Output root:', OUT_DIR.resolve())

POLICY_URL_RE = re.compile(r'^https?://das\.ohio\.gov/.+?/policies/.+', re.IGNORECASE)
PDF_URL_RE = re.compile(r'\.pdf(\?.*)?$', re.IGNORECASE)


def safe_filename(s: str, max_len: int = 120) -> str:
    s = re.sub(r'[^a-zA-Z0-9._-]+', '_', s.strip())
    return s[:max_len] if len(s) > max_len else s


def sha1(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()[:10]


def extract_pdf_text(pdf_path: str) -> str:
    try:
        reader = PdfReader(pdf_path)
        parts = []
        for i, page in enumerate(reader.pages):
            txt = page.extract_text() or ''
            txt = txt.strip()
            if txt:
                parts.append(f'\n--- PDF Page {i+1} ---\n{txt}\n')
        return '\n'.join(parts).strip()
    except Exception as e:
        return f'[PDF TEXT EXTRACTION ERROR] {e}'


def download_file(url: str, out_path: str, timeout: int = 60) -> bool:
    try:
        r = requests.get(url, stream=True, timeout=timeout, headers={'User-Agent': BROWSER_UA})
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)
        return True
    except Exception:
        return False


def clean_text_from_html(html: str) -> str:
    soup = BeautifulSoup(html, 'lxml')

    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()

    text = soup.get_text('\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = '\n'.join([line.rstrip() for line in text.splitlines()])
    return text.strip()


def extract_links_from_html(base_url: str, html: str) -> set[str]:
    soup = BeautifulSoup(html, 'lxml')
    urls = set()

    for a in soup.select('a[href]'):
        href = a.get('href', '').strip()
        if not href:
            continue
        urls.add(urljoin(base_url, href))

    for el in soup.select('[data-href], [data-url], [data-link]'):
        for attr in ['data-href', 'data-url', 'data-link']:
            v = el.get(attr)
            if v:
                urls.add(urljoin(base_url, v.strip()))

    for el in soup.select('[onclick]'):
        onclick = el.get('onclick') or ''
        for m in re.findall(r"(https?://[^\s\"']+)", onclick):
            urls.add(m)

    return set(u.split('#')[0] for u in urls)

Output root: C:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies


In [23]:
def collect_policy_links_from_policy_finder_sync() -> list[str]:
    """
    Run Playwright Sync API in a worker thread (safe for Jupyter).
    Also bumps table page size to 100 and paginates through all pages.
    """
    if sys.platform.startswith('win') and hasattr(asyncio, 'WindowsProactorEventLoopPolicy'):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    found = set()

    def collect_visible_policy_links(page_obj):
        anchors = page_obj.locator('a[href]').all()
        for a in anchors:
            href = a.get_attribute('href') or ''
            if not href:
                continue
            if href.startswith('/'):
                href = 'https://das.ohio.gov' + href
            href = href.split('#')[0].strip()
            if POLICY_URL_RE.match(href):
                found.add(href)

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.set_extra_http_headers({'User-Agent': BROWSER_UA})

        network_hits = []

        def on_response(resp):
            try:
                url = resp.url
                ct = (resp.headers.get('content-type') or '').lower()
                if 'application/json' in ct or 'text/json' in ct or 'application/vnd' in ct:
                    network_hits.append(url)
            except Exception:
                pass

        page.on('response', on_response)

        page.goto(START_URL, wait_until='domcontentloaded', timeout=120_000)
        page.wait_for_timeout(2500)

        # Scroll a little to trigger lazy-loaded table/widget initialization
        for _ in range(8):
            page.mouse.wheel(0, 1500)
            page.wait_for_timeout(250)

        # Try common DataTables/portal page-size controls to set 100 rows per page.
        page_size_locators = [
            "select[name$='_length']",
            "select[aria-label*='entries per page']",
            "label:has-text('entries per page') select",
            "select:near(:text('entries per page'))",
        ]
        for selector in page_size_locators:
            try:
                select = page.locator(selector).first
                if select.count() > 0:
                    options = select.locator('option').all_inner_texts()
                    normalized = {o.strip() for o in options}
                    if '100' in normalized:
                        select.select_option('100')
                    elif '100 ' in normalized:
                        select.select_option('100 ')
                    else:
                        # Best effort: set by index if first option is 25 and second is 50/100
                        select.select_option(index=min(2, max(0, len(options) - 1)))
                    page.wait_for_timeout(1200)
                    break
            except Exception:
                continue

        # Collect from current page + paginate via Next button.
        collect_visible_policy_links(page)
        max_pages = 200
        for _ in range(max_pages):
            next_button_selectors = [
                "a.paginate_button.next",
                "button[aria-label='Next']",
                "a:has-text('Next')",
                "li.next a",
            ]
            next_clicked = False
            for sel in next_button_selectors:
                try:
                    nxt = page.locator(sel).first
                    if nxt.count() == 0:
                        continue
                    cls = (nxt.get_attribute('class') or '').lower()
                    aria_disabled = (nxt.get_attribute('aria-disabled') or '').lower()
                    disabled = ('disabled' in cls) or (aria_disabled == 'true')
                    if disabled:
                        continue
                    if not nxt.is_visible():
                        continue
                    nxt.click(timeout=3000)
                    page.wait_for_timeout(1200)
                    collect_visible_policy_links(page)
                    next_clicked = True
                    break
                except Exception:
                    continue
            if not next_clicked:
                break

        # Also mine rendered HTML blobs.
        html = page.content()
        for u in extract_links_from_html('https://das.ohio.gov', html):
            if POLICY_URL_RE.match(u):
                found.add(u)

        # Mine JSON/network responses for policy links.
        for json_url in list(dict.fromkeys(network_hits))[:400]:
            try:
                r = page.request.get(json_url, timeout=60_000)
                if not r.ok:
                    continue
                body = r.text()
                for m in re.findall(r"https?://das\.ohio\.gov/[^\s\"']+/policies/[^\s\"']+", body):
                    found.add(m.split('#')[0])
            except Exception:
                continue

        browser.close()

    return sorted(found)


def collect_policy_links_fallback_requests() -> list[str]:
    found = set()
    candidate_pages = [
        'https://das.ohio.gov/employee-relations/policies',
        'https://das.ohio.gov/wps/portal/gov/das/employee-relations/policies',
        'https://das.ohio.gov/wps/portal/gov/das/home/policy-finder/filter-policy-finder',
    ]

    for page_url in candidate_pages:
        try:
            r = requests.get(page_url, timeout=60, headers={'User-Agent': BROWSER_UA})
            if r.status_code >= 400:
                continue
            links = extract_links_from_html(r.url or page_url, r.text)
            for link in links:
                if POLICY_URL_RE.match(link):
                    found.add(link)
        except Exception:
            continue

    return sorted(found)


async def collect_policy_links_from_policy_finder() -> list[str]:
    links = await asyncio.to_thread(collect_policy_links_from_policy_finder_sync)
    if links:
        return links

    print('Playwright found 0 links. Falling back to direct page parsing...')
    return await asyncio.to_thread(collect_policy_links_fallback_requests)


def crawl_policy_page(policy_url: str) -> dict:
    r = requests.get(policy_url, timeout=60, headers={'User-Agent': BROWSER_UA})
    r.raise_for_status()

    html = r.text
    soup = BeautifulSoup(html, 'lxml')

    title = (soup.title.get_text(' ', strip=True) if soup.title else '').strip()
    if not title:
        title = policy_url

    all_links = sorted(extract_links_from_html(policy_url, html))
    pdf_links = [u for u in all_links if PDF_URL_RE.search(u)]

    page_text = clean_text_from_html(html)

    pdf_text_blobs = []
    downloaded = []

    for pdf_url in pdf_links:
        pdf_name = safe_filename(os.path.basename(urlparse(pdf_url).path) or f'policy_{sha1(pdf_url)}.pdf')
        pdf_path = PDF_DIR / pdf_name

        if not pdf_path.exists():
            ok = download_file(pdf_url, str(pdf_path))
            if not ok:
                pdf_text_blobs.append(f'\n[PDF DOWNLOAD FAILED] {pdf_url}\n')
                continue

        downloaded.append({'pdf_url': pdf_url, 'pdf_path': str(pdf_path)})
        pdf_text = extract_pdf_text(str(pdf_path))
        pdf_text_blobs.append(f'\n\n========== PDF CONTENT ==========\nPDF: {pdf_url}\n\n{pdf_text}\n')

    references = all_links

    return {
        'url': policy_url,
        'title': title,
        'references': references,
        'pdfs': [d['pdf_url'] for d in downloaded],
        'page_text': page_text,
        'pdf_text': '\n'.join(pdf_text_blobs).strip(),
    }


def write_policy_txt(doc: dict) -> str:
    slug = safe_filename(urlparse(doc['url']).path.strip('/').split('/')[-1] or 'policy')
    fname = f"{slug}__{sha1(doc['url'])}.txt"
    out_path = TXT_DIR / fname

    lines = []
    lines.append(f"TITLE: {doc['title']}")
    lines.append(f"URL: {doc['url']}")
    lines.append('')
    lines.append('===== PAGE TEXT =====')
    lines.append(doc['page_text'] or '')
    lines.append('')

    lines.append('===== REFERENCES (ALL LINKS) =====')
    for u in doc['references']:
        lines.append(u)

    if doc['pdf_text']:
        lines.append('')
        lines.append(doc['pdf_text'])

    content = '\n'.join(lines).strip() + '\n'

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(content)

    return str(out_path)


async def main():
    print(f'Extracting policy links from: {START_URL}')
    policy_links = await collect_policy_links_from_policy_finder()
    print(f'Found {len(policy_links)} policy URLs')

    idx_path = OUT_DIR / 'policy_links.json'
    with open(idx_path, 'w', encoding='utf-8') as f:
        json.dump(policy_links, f, indent=2)

    if not policy_links:
        print('No policy URLs found. (Likely blocked or needs different selector/network mining.)')
        return

    successes = 0
    failures = []

    for i, url in enumerate(policy_links, 1):
        try:
            print(f'[{i}/{len(policy_links)}] Crawling: {url}')
            doc = crawl_policy_page(url)
            out_txt = write_policy_txt(doc)
            print('  wrote:', out_txt)
            successes += 1
            time.sleep(0.15)
        except Exception as e:
            failures.append({'url': url, 'error': str(e)})

    print(f'\nDONE. Success: {successes}  Failures: {len(failures)}')
    if failures:
        fail_path = OUT_DIR / 'failures.json'
        with open(fail_path, 'w', encoding='utf-8') as f:
            json.dump(failures, f, indent=2)
        print(f'Wrote failures to: {fail_path}')

    print(f'TXT files: {TXT_DIR}')
    print(f'PDF files: {PDF_DIR}')
    print(f'Index: {idx_path}')

In [24]:
# Execute crawl
await main()

Extracting policy links from: https://das.ohio.gov/home/policy-finder/filter-policy-finder
Found 152 policy URLs
[1/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/buying-and-selling/policies/copier-management-policy
  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies\txt\copier-management-policy__283cfc78c8.txt
[2/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/buying-and-selling/policies/debarment-policies
  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies\txt\debarment-policies__7c7f693763.txt
[3/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/buying-and-selling/policies/executive-order-2019-12D
  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies\txt\executive-order-2019-12D__e03116c1f0.txt
[4/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/buying-and-selling/policies/pm-02-emergency-purchasing-procedures
  wrote: c:\V

invalid pdf header: b'<!DOC'
EOF marker not found
invalid pdf header: b'<!DOC'
EOF marker not found


  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies\txt\it-17__2e77f1215d.txt
[139/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/technology-and-strategy/policies/it-21
  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies\txt\it-21__a786b7e3bb.txt
[140/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/technology-and-strategy/policies/its-plf-01
  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies\txt\its-plf-01__bd72d83ec1.txt
[141/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/technology-and-strategy/policies/its-plf-01-a
  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBooks\out\das_policies\txt\its-plf-01-a__cae8144d6f.txt
[142/152] Crawling: https://das.ohio.gov/wps/portal/gov/das/technology-and-strategy/policies/its-plf-01-b
  wrote: c:\VarunProjects\2026\MistrV\PolicyPlatform\policy-back\tools\NoteBo